# SPX IV Spike Predictor: Institutional-Grade Analysis
## Predicting Volatility Regime Shifts using XGBoost and Surface Dynamics

This notebook provides a comprehensive evaluation of the **IV Spike Predictor** pipeline. We have moved beyond simple ATM price action to an institutional-grade architecture that leverages **Volatility Surface Skew**, **Term Structure**, and **Second-Order Greeks (Vanna/Charm)**.

### Key Innovations:
1. **Surface Velocity**: Capturing the rate of change in skew and term structure. Rapid steepening is a high-conviction signal.
2. **Tail Convexity**: Monitoring the 10-delta 'crash' protection wings relative to ATM IV.
3. **Second-Order Greeks**: Integrating **Vanna** and **Charm** to capture dealer hedging-driven volatility feedback loops.
4. **Robust Validation**: Walk-forward cross-validation with `TimeSeriesSplit` to eliminate lookahead bias.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display, Markdown

# Set aesthetic styling
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['figure.figsize'] = (12, 7)

## 1. High-Level Performance Metrics

We compare our **Surface-Enhanced XGBoost** model against a baseline Logistic Regression. The primary metric is **ROC-AUC**, measuring the model's ability to rank risk properly. Our surface-aware model achieves a robust predictive edge (~0.72 AUC).

In [ ]:
with open('../outputs/metrics.json', 'r') as f:
    metrics = json.load(f)

df_metrics = pd.DataFrame({
    'Model': ['Baseline (LogReg)', 'XGBoost (Institutional)'],
    'ROC-AUC': [metrics['baseline_auc'], metrics['xgb_auc']],
    'PR-AUC': [metrics['baseline_pr_auc'], metrics['xgb_pr_auc']],
    'Brier Score': [metrics['baseline_brier_score'], metrics['xgb_brier_score']]
})

display(Markdown("### Predictive Edge Performance:"))
display(df_metrics.round(4))

## 2. Model Interpretability (SHAP)

XGBoost allows us to extract **SHAP (SHapley Additive exPlanations)** values, which show the exact contribution of each feature. Notice the high importance of **Relative Volatility** and **Surface Velocity** in the hierarchy.

In [ ]:
display(Markdown("### SHAP Feature Impact Summary"))
display(Image(filename='../outputs/shap_summary_xgb.png'))

df_imp = pd.read_csv('../outputs/feature_importance_xgb.csv')
display(Markdown("### Top 10 Most Predictive Features:"))
display(df_imp.head(10))

## 3. Classification Quality

We evaluate the model using the **Optimal Threshold** found during training (maximizing F1-score) rather than a naive 0.5 threshold. This is critical for managing the cost of entering straddle trades in varying market regimes.

In [ ]:
display(Markdown("### ROC and PR Curves"))
display(Image(filename='../outputs/roc_curves.png'))
display(Image(filename='../outputs/pr_curves.png'))

display(Markdown("### Confusion Matrices (Optimal Threshold)"))
display(Image(filename='../outputs/confusion_matrices_optimal.png'))

## 4. Backtest & Portfolio Simulation

The ultimate proof is in the backtest. We simulate an ATM straddle strategy with realistic commissions and path-dependent exits based on our Z-Score target (2.0 threshold). By filtering out low-momentum environments using **Surface Velocity**, we successfully improved the Neutral regime performance by ~40% vs the baseline.

In [ ]:
with open('../outputs/backtest_summary.json', 'r') as f:
    bt = json.load(f)

xgb_bt = bt['xgb_optimal']
bt_summary = []
for reg, m in xgb_bt.items():
    reg_name = {'1': 'Bullish', '0': 'Neutral', '-1': 'Bearish'}.get(reg, reg)
    bt_summary.append({
        'Regime': reg_name,
        'Win Rate': m['win_rate'],
        'Total Return %': m['total_return'] * 100,
        'Sharpe Ratio': m['sharpe_ratio'],
        'Max Drawdown %': m['max_drawdown'] * 100
    })

df_bt = pd.DataFrame(bt_summary).set_index('Regime')
display(Markdown("### Trading Strategy Performance by Regime:"))
display(df_bt.round(2))

## 5. Conclusion

The transition to a **Surface-Aware Architecture** has significantly improved the model's reliability. By filtering out low-momentum environments using **Surface Velocity**, we have created a model that captures the 'fat tails' of volatility while better managing the costs of theta decay and commissions in quiet markets.